## Part I: Fetching Data

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import timedelta
import warnings

# Suppress noisy yfinance and pandas warnings
warnings.filterwarnings('ignore')

In [ ]:
SCORES_URL   = "https://raw.githubusercontent.com/darrentweng/wharton-uss-academic-journal-executive-sentiment/refs/heads/main/data/strux_scores_clean.csv"
MANIFEST_URL = "https://raw.githubusercontent.com/darrentweng/wharton-uss-academic-journal-executive-sentiment/refs/heads/main/data/strux_manifest_clean.csv"

manifest = pd.read_csv(MANIFEST_URL)
scores   = pd.read_csv(SCORES_URL)

# Merge early so we only compute what we strictly need
df = pd.merge(scores, manifest, on=["ticker", "date"], how="inner")
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset merged: {len(df)} rows, {df['ticker'].nunique()} tickers")


In [ ]:
unique_tickers = df['ticker'].unique().tolist()
# Add a generous 45-day buffer for 10-day windows and weekend gaps
min_date = (df["date"].min() - timedelta(days=45)).strftime("%Y-%m-%d")
max_date = (df["date"].max() + timedelta(days=45)).strftime("%Y-%m-%d")

print(f"Pre-fetching data for {len(unique_tickers)} tickers from {min_date} to {max_date}...\n")

# Fetch SPY
spy_raw = yf.download("SPY", start=min_date, end=max_date, progress=False)
spy_rets = np.log(spy_raw["Close"].squeeze() / spy_raw["Close"].squeeze().shift(1)).dropna()
spy_rets.index = pd.to_datetime(spy_rets.index).tz_localize(None)

In [ ]:
# Bulk fetch all unique tickers in ONE network call
print("Downloading continuous prices for all tickers at once (this is fast)...")
bulk_data = yf.download(unique_tickers, start=min_date, end=max_date, progress=False)
bulk_prices = bulk_data["Close"]

In [ ]:
price_cache = {}
returns_cache = {}

for ticker in unique_tickers:
    try:
        # Handle dataframe structure depending on if it's 1 ticker or multiple
        prices = bulk_prices[ticker].dropna() if isinstance(bulk_prices, pd.DataFrame) else bulk_prices.dropna()
        prices.index = pd.to_datetime(prices.index).tz_localize(None)

        price_cache[ticker] = prices
        returns_cache[ticker] = np.log(prices / prices.shift(1)).dropna()
    except Exception:
        price_cache[ticker], returns_cache[ticker] = None, None

In [ ]:
print("Caching earnings dates (this takes a moment)...")
earnings_cache = {}
for i, ticker in enumerate(unique_tickers):
    if (i + 1) % 50 == 0:
        print(f"  Fetched earnings for {i + 1}/{len(unique_tickers)} tickers...")
    try:
        ed = yf.Ticker(ticker).earnings_dates
        if ed is not None and not ed.empty:
            ed.index = pd.to_datetime(ed.index).tz_localize(None)
            earnings_cache[ticker] = ed
        else:
            earnings_cache[ticker] = None
    except Exception:
        earnings_cache[ticker] = None

print("\nPre-fetching complete! All required market data is now in RAM.")

## Part II: Computing Metrics

In [ ]:
def calculate_all_metrics(row):
    ticker    = row['ticker']
    call_date = row['date']

    metrics = {
        'volatility_pre_10d':  None,
        'volatility_post_10d': None,
        'volatility_pre_5d':   None,
        'volatility_post_5d':  None,
        'volatility_pre_20d':  None,
        'volatility_post_20d': None,
        'n_pre': 0, 'n_post': 0, 'vol_error': None,
        'eps_surprise': None, 'abnormal_return': None,
    }

    prices = price_cache.get(ticker)
    rets   = returns_cache.get(ticker)
    ed     = earnings_cache.get(ticker)

    # ── Volatility ──────────────────────────────────────────
    if rets is not None and not rets.empty:
        for window, min_days in [(10, 5), (5, 3), (20, 10)]:
            pre  = rets[rets.index < call_date].iloc[-window:]
            post = rets[rets.index > call_date].iloc[:window]

            if window == 10:
                metrics['n_pre']  = len(pre)
                metrics['n_post'] = len(post)

            if len(pre) >= min_days:
                metrics[f'volatility_pre_{window}d']  = float(np.std(pre,  ddof=1))
            elif window == 10:
                metrics['vol_error'] = f"Insufficient pre-window ({len(pre)} days)"

            if len(post) >= min_days:
                metrics[f'volatility_post_{window}d'] = float(np.std(post, ddof=1))
            elif window == 10 and not metrics['vol_error']:
                metrics['vol_error'] = f"Insufficient post-window ({len(post)} days)"
    else:
        metrics['vol_error'] = "No price data"

    # ── EPS Surprise ────────────────────────────────────────
    if ed is not None and not ed.empty and prices is not None and not prices.empty:
        temp_ed = ed.copy()
        temp_ed["diff"] = np.abs((temp_ed.index - call_date).days)
        closest = temp_ed[temp_ed["diff"] <= 5].sort_values("diff")

        if not closest.empty:
            actual    = closest.iloc[0].get("Reported EPS")
            estimated = closest.iloc[0].get("EPS Estimate")

            if not pd.isna(actual) and not pd.isna(estimated):
                if call_date in prices.index:
                    share_price = float(prices.loc[call_date])
                else:
                    diffs       = np.abs((prices.index - call_date).days)
                    nearest_idx = diffs.argmin()
                    share_price = float(prices.iloc[nearest_idx]) if diffs[nearest_idx] <= 3 else None

                if share_price:
                    metrics['eps_surprise'] = (actual - estimated) / share_price

    # ── Abnormal Return ─────────────────────────────────────
    if rets is not None and not rets.empty:
        target = call_date
        if target not in rets.index:
            diffs  = np.abs((rets.index - target).days)
            target = rets.index[diffs.argmin()] if diffs.min() <= 2 else None

        if target is not None and target in spy_rets.index:
            metrics['abnormal_return'] = (
                float(rets.loc[target]) - float(spy_rets.loc[target])
            )

    return pd.Series(metrics)

In [ ]:
# Apply functions locally
print("\nComputing metrics for all rows...")
metrics_df = df.apply(calculate_all_metrics, axis=1)

# Join the metrics back to the main dataframe
df = pd.concat([df, metrics_df], axis=1)

# Combine EPS and Abnormal return to get the Surprise Factor
df["surprise_factor"] = df["eps_surprise"].combine_first(df["abnormal_return"])

# Summary
print(f"\nFinal master dataset summary:")
print(f"  Total rows:              {len(df)}")
print(f"  Unique tickers:          {df['ticker'].nunique()}")
print(f"  Date range:              {df['date'].min().date()} to {df['date'].max().date()}")
print(f"  EPS surprise populated:  {df['eps_surprise'].notna().sum()} ({df['eps_surprise'].notna().mean()*100:.1f}%)")
print(f"  Abnormal return:         {df['abnormal_return'].notna().sum()} ({df['abnormal_return'].notna().mean()*100:.1f}%)")
print(f"  Surprise factor:         {df['surprise_factor'].notna().sum()} ({df['surprise_factor'].notna().mean()*100:.1f}%)")
print(f"  Missing surprise_factor: {df['surprise_factor'].isna().sum()}")
print(f"  Missing vol_post_10d:    {df['volatility_post_10d'].isna().sum()}")
print(f"  Vol errors:              {df['vol_error'].notna().sum()}")

# Save final dataset
df['date'] = df['date'].dt.strftime('%Y-%m-%d')
df.to_csv("/content/master_dataset.csv", index=False)
print("\nSaved: /content/master_dataset.csv")
print("Ready for regression notebook.")